## Transitions for offline training

An environment built with `record="transitions"` keeps a row for each step each of its
accounts takes: its episode and step, the observation at the close, the action as given,
the reward, both flags and the observation the step returned. `export` writes them as one
Arrow file, with the spaces, the reward and the agent spec on its schema, which any Arrow
reader opens. A backtest of an agent records the same rows at its own level.

In [1]:
import json
import pathlib

import exchange_router
import numpy
import pandas
import probatum
import pyarrow
import pyarrow.ipc

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}, "
      f"pyarrow {pyarrow.__version__}")

probatum 3.0.0, exchange-router 3.3.0, pyarrow 26.0.1


### The bars

In [2]:
PATH = pathlib.Path("../bars/bybit-linear-btcusdt-ethusdt-1h-2024-2025")

if PATH.exists():
    perp = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["bybit"]) as router:
        perp = probatum.Data.from_router(router, exchange="bybit", market="linear",
                                         interval="1h", symbols=["BTCUSDT", "ETHUSDT"],
                                         start="2024-01-01", end="2025-12-31", funding=True)
    perp.save(path=PATH)

train, test = perp["BTCUSDT"].split(when="2025-07-01")

### Rollouts, recorded

Sixteen accounts in episodes of a week, each step's action the sign of the last day's
return, which the market block holds as the closes over the last close.

In [3]:
env = probatum.env(train, venue="bybit_perp", window=24, num_envs=16,
                   episode_length=24 * 7, record="transitions", seed=0)

obs, _ = env.reset(seed=0)
for _ in range(500):
    day     = 1.0 / obs[:, 3] - 1.0
    actions = numpy.sign(day)[:, None].astype(numpy.float32)
    obs, *_ = env.step(actions)

rows = env.transitions

print(rows)

pandas.DataFrame({name: rows[name] for name in ["env", "episode", "step", "reward",
                                                "terminated", "truncated"]}).head(n=4)

Transitions(7,966 rows, observations of 128 values, actions of 1)


,env,episode,step,reward,terminated,truncated
0,0,0,0,0.003773,False,False
1,1,0,0,-0.001258,False,False
2,2,0,0,-0.002502,False,False
3,3,0,0,0.001819,False,False


### One Arrow file

Written into `saved/`, which git ignores, and read back with pyarrow alone.

In [4]:
saved = pathlib.Path("../saved/rollouts.arrow")
saved.parent.mkdir(exist_ok=True)

env.export(format="arrow", path=saved)

table = pyarrow.ipc.open_file(saved).read_all()

print(f"{saved.name}, {saved.stat().st_size / 2**20:.1f} MB, {table.num_rows:,} rows")
print(table.schema.to_string(show_schema_metadata=False))

rollouts.arrow, 8.1 MB, 7,966 rows
env: int64 not null
episode: int64 not null
step: int64 not null
observation: fixed_size_list<item: float not null>[128] not null
  child 0, item: float not null
action: fixed_size_list<item: float not null>[1] not null
  child 0, item: float not null
reward: double not null
terminated: bool not null
truncated: bool not null
next_observation: fixed_size_list<item: float not null>[128] not null
  child 0, item: float not null


### What the schema carries

The spaces the rows were recorded in, the reward, and the agent spec, as JSON text on the
schema, so a file says how its rows were made.

In [5]:
meta = {key.decode(): value for key, value in table.schema.metadata.items()}

print(sorted(meta))

json.loads(meta["probatum.spaces"])

['probatum.reward', 'probatum.spaces', 'probatum.spec', 'probatum.version']


{'observation': {'size': 128,
  'layout': 'flat',
  'shape': [24, 1, 5],
  'account': ['weight',
   'upnl',
   'entry_distance',
   'shown',
   'log_return',
   'leverage',
   'margin_ratio',
   'cash_weight']},
 'action': {'space': 'box', 'low': [-1.0], 'high': 1.0, 'max_leverage': 1.0}}

### A backtest's transitions

The same rule as a function of the observation, backtested on the bars after training with
`record="transitions"`: a row a close it was asked at, until its episode ended.

In [6]:
def by_the_day(observation):
    return numpy.sign(1.0 / observation[:, 3] - 1.0)[:, None].astype(numpy.float32)


result = probatum.backtest(by_the_day, data=test, venue="bybit_perp", spec=env,
                           record="transitions")

result.export(format="arrow", path="../saved/test.arrow")

print(result)
print(result.transitions)

Result(the function by_the_day on probatum.venues.bybit_perp, 4,416 bars, equity 10,000.00 to 4,214.63, recorded at 'transitions')
Transitions(4,415 rows, observations of 128 values, actions of 1)
